# Week 4 L1 — Activity 3b: Merging two tables

A **merge** (also called a *join*) puts two tables side by side by matching rows that share the same **key**. It answers questions neither table can answer alone: how many appointments were there *per 1,000 residents*? The appointments are in one file; the residents are in the other.

We use both clean files from Activity 1:

| File | Grain (one row is…) |
|---|---|
| `community_health_activity_clean.csv` | one area × year × age group × sex |
| `area_population_clean.csv` | one area × year |

Merging safely comes down to four questions, which this notebook works through in order:

1. **What is the key?** Which columns identify the matching rows?
2. **Is the key unique** on the side that should have one row per key?
3. **What happens to rows that don't match?**
4. **Did the number of rows change the way I expected?**

In [3]:
import pandas as pd

activity = pd.read_csv("community_health_activity_clean.csv")
activity["appointments"] = activity["appointments"].astype("Int64")

population = pd.read_csv("area_population_clean.csv")
population["population"] = population["population"].astype("Int64")

print("activity:  ", activity.shape)
print("population:", population.shape)

activity:   (12, 9)
population: (6, 6)


In [2]:
population

,area_code,area_name,year,population,region,deprivation_band
0,CH001,Northside,2024,52400,North,Low
1,CH001,Northside,2025,53100,North,Low
2,CH002,Riverside,2024,48750,North,Medium
3,CH002,Riverside,2025,49200,North,Medium
4,CH003,Hilltown,2024,36900,North,High
5,CH003,Hilltown,2025,<NA>,North,High


---
## 1. Find the key

A row of `activity` should match the population row for **the same area in the same year**. So the key is two columns: area code and year.

But the year column has a different name in each file:

In [3]:
print("activity columns:  ", list(activity.columns))
print("population columns:", list(population.columns))

activity columns:   ['area_code', 'area_name', 'reporting_year', 'age_group', 'sex', 'appointments', 'average_wait_days', 'follow_up_needed', 'notes']
population columns: ['area_code', 'area_name', 'year', 'population', 'region', 'deprivation_band']


`activity` calls it `reporting_year`; `population` calls it `year`. The simplest fix is to rename one so the key columns have the **same name in both tables** (*Activity 2, Step 3*).

In [4]:
population = population.rename(columns={"year": "reporting_year"})

key = ["area_code", "reporting_year"]

---
## 2. Check the key is unique on the "one" side

Each area-year has **one** population figure but **several** activity rows (one per age group and sex). This is called a **many-to-one** merge: many activity rows → one population row.

If the population table accidentally had the same area-year twice, every matching activity row would be copied twice, and nothing would warn you. So check first (*Activity 1, Step 9*):

In [5]:
print("Repeated keys in population:", population.duplicated(key).sum())

Repeated keys in population: 0


---
## 3. The merge

`pd.merge(left, right, on=key, how=...)` matches rows of `left` with rows of `right` that have the same key.

- `on=key` — the columns to match on
- `how="inner"` — keep only rows that found a match on **both** sides
- `validate="many_to_one"` — pandas **checks** that the right-hand key is unique, and stops with an error if it is not. It turns Section 2 into a guarantee.

In [6]:
merged = pd.merge(activity, population, on=key, how="inner", validate="many_to_one")

print("Rows before:", len(activity))
print("Rows after: ", len(merged))
merged

Rows before: 12
Rows after:  12


,area_code,area_name_x,reporting_year,age_group,sex,appointments,average_wait_days,follow_up_needed,notes,area_name_y,population,region,deprivation_band
0,CH001,Northside,2024,0-17,Female,1250,7.0,Yes,school clinics,Northside,52400,North,Low
1,CH001,Northside,2024,18-64,Male,980,5.0,No,NaN,Northside,52400,North,Low
2,CH001,Northside,2024,65+,Female,430,7.5,Yes,telephone follow-up,Northside,52400,North,Low
3,CH001,Northside,2025,0-17,Female,1280,6.0,No,NaN,Northside,53100,North,Low
4,CH002,Riverside,2024,0-17,Female,875,6.0,No,NaN,Riverside,48750,North,Medium
5,CH002,Riverside,2024,18-64,Male,1040,8.0,Yes,evening service,Riverside,48750,North,Medium
6,CH002,Riverside,2024,65+,Male,310,7.5,No,data-entry error in wait,Riverside,48750,North,Medium
7,CH002,Riverside,2025,18-64,Unknown,1060,7.0,Unknown,sex and follow-up missing,Riverside,49200,North,Medium
8,CH003,Hilltown,2024,0-17,Female,620,9.0,Yes,NaN,Hilltown,36900,North,High
9,CH003,Hilltown,2024,18-64,Male,900,10.0,No,NaN,Hilltown,36900,North,High


12 rows in, 12 rows out: every activity row found its population row, and none was copied.

### The `_x` and `_y` columns

Look at the column names. Both tables had an `area_name` column that was **not** part of the key, so pandas kept both and added `_x` (from the left table) and `_y` (from the right table):

In [7]:
list(merged.columns)

['area_code',
 'area_name_x',
 'reporting_year',
 'age_group',
 'sex',
 'appointments',
 'average_wait_days',
 'follow_up_needed',
 'notes',
 'area_name_y',
 'population',
 'region',
 'deprivation_band']

The names are the same in both files, so we need only one. The cleanest fix is to drop the extra column **before** merging, so only the left table supplies it:

In [8]:
population_for_merge = population.drop(columns=["area_name"])

merged = pd.merge(activity, population_for_merge, on=key, how="inner", validate="many_to_one")
list(merged.columns)

['area_code',
 'area_name',
 'reporting_year',
 'age_group',
 'sex',
 'appointments',
 'average_wait_days',
 'follow_up_needed',
 'notes',
 'population',
 'region',
 'deprivation_band']

---
## 4. When rows don't match

So far every row matched. In real data some won't. To see what happens, pretend we only have population figures for **2024**:

In [9]:
population_2024 = population_for_merge[population_for_merge["reporting_year"] == 2024]
population_2024

,area_code,reporting_year,population,region,deprivation_band
0,CH001,2024,52400,North,Low
2,CH002,2024,48750,North,Medium
4,CH003,2024,36900,North,High


The `how` setting decides what happens to activity rows from 2025, which now have no partner:

| `how=` | Keeps | Rows with no match |
|---|---|---|
| `"inner"` | only rows that match on both sides | **dropped** |
| `"left"` | every row of the left table | kept, with blanks for the right table's columns |
| `"right"` | every row of the right table | kept, with blanks for the left table's columns |
| `"outer"` | every row of both tables | kept from both sides, with blanks |

In [10]:
inner = pd.merge(activity, population_2024, on=key, how="inner")
left = pd.merge(activity, population_2024, on=key, how="left")

print("inner:", len(inner), "rows")
print("left: ", len(left), "rows")

inner: 9 rows
left:  12 rows


The inner merge **silently dropped** 3 rows. Nothing warned us. This is the most common way data goes missing during analysis.

### Seeing the unmatched rows with `indicator=True`

`indicator=True` adds a column called `_merge` saying where each row came from: `both`, `left_only` or `right_only`. Use it with `how="outer"` to see **everything** that did and didn't match:

In [11]:
check = pd.merge(activity, population_2024, on=key, how="outer", indicator=True)
check["_merge"].value_counts()

_merge
both          9
left_only     3
right_only    0
Name: count, dtype: int64

In [12]:
check[check["_merge"] == "left_only"][["area_code", "reporting_year", "age_group", "sex", "appointments"]]

,area_code,reporting_year,age_group,sex,appointments
3,CH001,2025,0-17,Female,1280
7,CH002,2025,18-64,Unknown,1060
11,CH003,2025,65+,Male,<NA>


Now you can **report** the loss: "3 activity rows from 2025 had no population figure and were excluded." That sentence belongs in your write-up.

> **Habit.** Before a real merge, run an outer merge with `indicator=True` and look at the `left_only` and `right_only` rows. Then decide, on purpose, whether to drop them.

---
## 5. Checking keys before you merge

You can also find unmatched keys without merging, using `.isin()` (*Activity 2*). For example: which area codes in `activity` do not appear in `population` at all?

In [13]:
codes_in_population = population["area_code"]
in_both = activity["area_code"].isin(codes_in_population)

print("Activity rows whose area is missing from population:", (~in_both).sum())

Activity rows whose area is missing from population: 0


Zero here, because the files cover the same three areas. With two real files from different sources, expect this number to be larger than zero, and expect to explain why.

---
## 6. Why merge? A new measure

Now each row carries both appointments and population, so we can calculate a **rate**: appointments per 1,000 residents.

In [14]:
merged["appointments_per_1000"] = merged["appointments"] / merged["population"] * 1000
merged["appointments_per_1000"] = merged["appointments_per_1000"].round(1)

merged[["area_code", "reporting_year", "age_group", "sex", "appointments", "population", "appointments_per_1000"]]

,area_code,reporting_year,age_group,sex,appointments,population,appointments_per_1000
0,CH001,2024,0-17,Female,1250,52400,23.9
1,CH001,2024,18-64,Male,980,52400,18.7
2,CH001,2024,65+,Female,430,52400,8.2
3,CH001,2025,0-17,Female,1280,53100,24.1
4,CH002,2024,0-17,Female,875,48750,17.9
5,CH002,2024,18-64,Male,1040,48750,21.3
6,CH002,2024,65+,Male,310,48750,6.4
7,CH002,2025,18-64,Unknown,1060,49200,21.5
8,CH003,2024,0-17,Female,620,36900,16.8
9,CH003,2024,18-64,Male,900,36900,24.4


CH003 in 2025 has no rate, because both its appointment count and its population are missing. Missing in, missing out: that is correct.

---
## 7. A trap after many-to-one merges

The population of an area-year now appears on **every** activity row for that area-year. Add it up naively and you count the same people several times:

In [15]:
print("Real total population (from the population file):", population["population"].sum())
print("Naive total from the merged table:              ", merged["population"].sum())

Real total population (from the population file): 240350
Naive total from the merged table:               516450


The merged figure is far too big because CH001's 2024 population was counted three times, once per activity row. After a many-to-one merge, the "one" side's numbers are **repeated**, so they must not be summed across rows.

> **Question.** If you needed the total population per year, which table would you use, and why?

*Your answer:*

---
## What you used today

| Tool | What it does |
|---|---|
| `.rename(columns={...})` | give key columns the same name in both tables |
| `.duplicated(key).sum()` | check the key is unique before merging |
| `pd.merge(left, right, on=key, how="inner")` | join two tables on a key |
| `validate="many_to_one"` | make pandas check the right-hand key is unique |
| `how="left"` / `"outer"` | keep unmatched rows instead of dropping them |
| `indicator=True` + `_merge` | see which rows matched and which did not |
| `.isin()` | check which keys exist in another table |
| `.drop(columns=[...])` before merging | avoid `_x` / `_y` duplicate columns |

## Exit ticket

1. What was the key, and what kind of merge was it (one-to-one, many-to-one…)?
2. How many rows did the inner merge drop when only 2024 population was available? How did you find out?
3. Why is summing `population` in the merged table wrong?

In [ ]:
merged.to_csv("data/community_health_activity_with_population.csv", index=False)